In [8]:
"""
dedup_via_temp.py
=================
Chiến lược bảng tạm để dedup ingredients (Loại 1 — chỉ khác case).

Flow:
  1. Tạo bảng tạm ingredients_temp
  2. Group các tên trùng case → chọn canonical (dùng nhiều nhất)
  3. Lowercase tên canonical → insert vào ingredients_temp
  4. Xây dựng mapping: old_id → canonical_id
  5. Cập nhật dish_ingredient theo mapping
  6. Truncate ingredients gốc
  7. Insert lại từ ingredients_temp
  8. Verify

Đổi DB_PATH trước khi chạy.
"""

import sqlite3
import logging
from pathlib import Path
from collections import defaultdict

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

DB_PATH  = "./cookpad_clean.db"   # ← ĐỔI ĐƯỜNG DẪN
DRY_RUN  = False                          # ← True để xem trước, không ghi DB


# ─────────────────────────────────────────────
# BƯỚC 1: Lấy tất cả ingredients + usage count
# ─────────────────────────────────────────────
def load_ingredients(conn: sqlite3.Connection) -> list[dict]:
    rows = conn.execute("""
        SELECT
            i.*,
            COUNT(di.id) AS usage_count
        FROM ingredients i
        LEFT JOIN dish_ingredient di ON di.ingredient_id = i.id
        GROUP BY i.id
        ORDER BY i.id
    """).fetchall()
    log.info(f"Loaded {len(rows)} ingredients từ DB.")
    return [dict(r) for r in rows]


# ─────────────────────────────────────────────
# BƯỚC 2: Group theo lowercase → chọn canonical
# ─────────────────────────────────────────────
def build_canonical_map(ingredients: list[dict]) -> tuple[list[dict], dict[int, int]]:
    """
    Trả về:
      canonical_rows  : list[dict] — các dòng canonical (đã lowercase tên)
      old_to_canonical: dict       — { old_id: canonical_id }
    """
    # Group theo tên lowercase
    groups: dict[str, list[dict]] = defaultdict(list)
    for ing in ingredients:
        key = ing["name"].strip().lower()
        groups[key].append(ing)

    canonical_rows   = []
    old_to_canonical = {}
    duplicate_count  = 0

    for lower_name, members in groups.items():
        # Canonical = dùng nhiều nhất, tie-break = id nhỏ nhất
        canonical = max(members, key=lambda x: (x["usage_count"], -x["id"]))

        # Gán tên lowercase cho canonical
        canonical_copy = dict(canonical)
        canonical_copy["name"] = lower_name

        canonical_rows.append(canonical_copy)

        # Map tất cả members (kể cả canonical) về canonical_id
        for m in members:
            old_to_canonical[m["id"]] = canonical["id"]

        if len(members) > 1:
            duplicate_count += 1
            log.info(
                f"  MERGE '{lower_name}': "
                + ", ".join(f"[{m['id']}]{m['name']}({m['usage_count']}x)" for m in members)
                + f" → canonical=[{canonical['id']}]"
            )

    log.info(
        f"Kết quả: {len(ingredients)} → {len(canonical_rows)} ingredients "
        f"({duplicate_count} nhóm trùng được merge)."
    )
    return canonical_rows, old_to_canonical


# ─────────────────────────────────────────────
# BƯỚC 3: Tạo bảng tạm và insert canonical rows
# ─────────────────────────────────────────────
def create_temp_table(conn: sqlite3.Connection, canonical_rows: list[dict]):
    # Lấy schema gốc của ingredients
    schema = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table' AND name='ingredients'"
    ).fetchone()["sql"]

    # Tạo bảng tạm với cùng schema, đổi tên
    temp_schema = schema.replace(
        'CREATE TABLE "ingredients"',
        'CREATE TABLE IF NOT EXISTS "ingredients_temp"'
    ).replace(
        "CREATE TABLE ingredients",
        'CREATE TABLE IF NOT EXISTS "ingredients_temp"'
    )

    # Xóa bảng tạm cũ nếu còn sót
    conn.execute("DROP TABLE IF EXISTS ingredients_temp")
    conn.execute(temp_schema)
    log.info("Đã tạo bảng ingredients_temp.")

    # Lấy danh sách cột thực tế (trừ usage_count ta tự thêm)
    col_info = conn.execute("PRAGMA table_info(ingredients)").fetchall()
    columns  = [c["name"] for c in col_info]

    placeholders = ", ".join("?" * len(columns))
    col_names    = ", ".join(f'"{c}"' for c in columns)

    insert_sql = f'INSERT INTO ingredients_temp ({col_names}) VALUES ({placeholders})'

    rows_to_insert = []
    for row in canonical_rows:
        rows_to_insert.append(tuple(row.get(c) for c in columns))

    conn.executemany(insert_sql, rows_to_insert)
    log.info(f"Đã insert {len(rows_to_insert)} canonical rows vào ingredients_temp.")


# ─────────────────────────────────────────────
# BƯỚC 4: Cập nhật dish_ingredient
# ─────────────────────────────────────────────
def update_dish_ingredient(conn: sqlite3.Connection, old_to_canonical: dict[int, int]):
    """
    Chỉ update những dòng mà ingredient_id trỏ về id bị deprecated
    (tức là old_id != canonical_id).
    """
    updates = [
        (canonical_id, old_id)
        for old_id, canonical_id in old_to_canonical.items()
        if old_id != canonical_id
    ]

    if not updates:
        log.info("Không có dish_ingredient nào cần cập nhật.")
        return

    conn.executemany(
        "UPDATE dish_ingredient SET ingredient_id = ? WHERE ingredient_id = ?",
        updates
    )
    log.info(f"Đã cập nhật dish_ingredient cho {len(updates)} ingredient_id bị deprecated.")


# ─────────────────────────────────────────────
# BƯỚC 5: Truncate gốc → Insert lại từ temp
# ─────────────────────────────────────────────
def swap_tables(conn: sqlite3.Connection):
    # Lấy danh sách cột
    col_info  = conn.execute("PRAGMA table_info(ingredients)").fetchall()
    columns   = [c["name"] for c in col_info]
    col_names = ", ".join(f'"{c}"' for c in columns)

    # SQLite không có TRUNCATE — dùng DELETE
    conn.execute("DELETE FROM ingredients")
    log.info("Đã xóa toàn bộ dữ liệu trong ingredients.")

    # Insert lại từ temp
    conn.execute(f"""
        INSERT INTO ingredients ({col_names})
        SELECT {col_names} FROM ingredients_temp
    """)
    count = conn.execute("SELECT COUNT(*) FROM ingredients").fetchone()[0]
    log.info(f"Đã insert lại {count} dòng canonical vào ingredients.")

    # Xóa bảng tạm
    conn.execute("DROP TABLE IF EXISTS ingredients_temp")
    log.info("Đã xóa ingredients_temp.")


# ─────────────────────────────────────────────
# BƯỚC 6: Verify
# ─────────────────────────────────────────────
def verify(conn: sqlite3.Connection):
    # Kiểm tra còn trùng case không
    remaining = conn.execute("""
        SELECT LOWER(name) as lower_name, COUNT(*) as cnt
        FROM ingredients
        GROUP BY LOWER(name)
        HAVING COUNT(*) > 1
        LIMIT 10
    """).fetchall()

    if remaining:
        log.warning(f"⚠️  Còn {len(remaining)} nhóm trùng case — kiểm tra lại!")
        for r in remaining:
            log.warning(f"  '{r['lower_name']}' xuất hiện {r['cnt']} lần")
    else:
        log.info("✅ Không còn trùng lặp case nào trong ingredients.")

    # Thống kê
    total_ing = conn.execute("SELECT COUNT(*) FROM ingredients").fetchone()[0]
    total_di  = conn.execute("SELECT COUNT(*) FROM dish_ingredient").fetchone()[0]
    log.info(f"ingredients: {total_ing} dòng active.")
    log.info(f"dish_ingredient: {total_di} dòng.")

    # Kiểm tra orphan — dish_ingredient trỏ về ingredient không tồn tại
    orphan = conn.execute("""
        SELECT COUNT(*) as cnt
        FROM dish_ingredient di
        LEFT JOIN ingredients i ON di.ingredient_id = i.id
        WHERE i.id IS NULL
    """).fetchone()["cnt"]

    if orphan > 0:
        log.warning(f"⚠️  {orphan} dish_ingredient trỏ về ingredient không tồn tại!")
    else:
        log.info("✅ Tất cả dish_ingredient đều trỏ về ingredient hợp lệ.")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    db_path = Path(DB_PATH)
    if not db_path.exists():
        log.error(f"Không tìm thấy DB: {db_path}")
        return

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    try:
        # Bật foreign key để SQLite check constraint
        conn.execute("PRAGMA foreign_keys = OFF")

        # B1: Load
        ingredients = load_ingredients(conn)

        # B2: Build canonical map
        canonical_rows, old_to_canonical = build_canonical_map(ingredients)

        if DRY_RUN:
            print("\n[DRY RUN] Không ghi DB. Xem log để kiểm tra kết quả.")
            return

        # B3: Tạo bảng tạm + insert canonical
        create_temp_table(conn, canonical_rows)

        # B4: Cập nhật dish_ingredient
        update_dish_ingredient(conn, old_to_canonical)

        # B5: Swap — truncate gốc, insert từ temp
        swap_tables(conn)

        conn.commit()
        log.info("✅ Commit thành công.")

        # B6: Verify
        print("\n── Verification ──")
        verify(conn)

    except Exception as e:
        conn.rollback()
        log.error(f"Lỗi — đã rollback: {e}")
        raise

    finally:
        conn.execute("PRAGMA foreign_keys = ON")
        conn.close()


main()

2026-03-08 10:20:35,332 [INFO] Loaded 1432 ingredients từ DB.
2026-03-08 10:20:35,337 [INFO]   MERGE 'nấm mèo': [1]Nấm mèo(0x), [26]nấm mèo(53x) → canonical=[26]
2026-03-08 10:20:35,337 [INFO]   MERGE 'hành lá': [16]hành lá(1303x), [209]Hành lá(0x) → canonical=[16]
2026-03-08 10:20:35,337 [INFO]   MERGE 'rau răm': [21]rau răm(186x), [2139]Rau răm(0x) → canonical=[21]
2026-03-08 10:20:35,339 [INFO]   MERGE 'ớt': [24]ớt(1014x), [1723]Ớt(0x) → canonical=[24]
2026-03-08 10:20:35,339 [INFO]   MERGE 'bông cải': [25]Bông cải(103x), [1607]bông cải(0x) → canonical=[25]
2026-03-08 10:20:35,340 [INFO]   MERGE 'dứa': [28]Dứa(95x), [330]dứa(0x) → canonical=[28]
2026-03-08 10:20:35,340 [INFO]   MERGE 'gừng': [39]Gừng(560x), [320]gừng(0x) → canonical=[39]
2026-03-08 10:20:35,340 [INFO]   MERGE 'cua': [41]cua(140x), [1579]Cua(0x) → canonical=[41]
2026-03-08 10:20:35,340 [INFO]   MERGE 'cà rốt': [42]Cà rốt(0x), [81]cà rốt(877x) → canonical=[81]
2026-03-08 10:20:35,340 [INFO]   MERGE 'thơm': [48]thơm(20


── Verification ──


In [ ]:
"""
aggregate_recipe_fields.py
==========================
Cập nhật bảng `recipes` với các field tổng hợp từ
dish_ingredient × ingredients_new (Nhóm 1).

Chạy: python aggregate_recipe_fields.py --db path/to/your.db
"""

import sqlite3
import json
import argparse
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

# ──────────────────────────────────────────────
# 1. SCHEMA: Danh sách cột cần thêm vào recipes
# ──────────────────────────────────────────────
NEW_COLUMNS = [
    ("dish_thermogenic_score", "REAL"),
    ("dish_hydration_score",   "REAL"),
    ("dish_warming_score",     "REAL"),
    ("dish_cooling_score",     "REAL"),
    ("dish_satiety_score",     "REAL"),
    ("dish_energy_total",      "REAL"),   # kcal / serving
    ("dish_sodium_total",      "REAL"),   # mg / serving
    ("dish_glycemic_load",     "REAL"),
    ("allergen_summary",       "TEXT"),   # JSON array
    ("is_vegan",               "INTEGER"),  # 0 / 1
    ("is_vegetarian",          "INTEGER"),  # 0 / 1
]

# ──────────────────────────────────────────────
# 2. ALTER TABLE – thêm cột nếu chưa có
# ──────────────────────────────────────────────
def migrate_schema(conn: sqlite3.Connection):
    cur = conn.execute("PRAGMA table_info(recipes)")
    existing = {row[1] for row in cur.fetchall()}

    added = []
    for col_name, col_type in NEW_COLUMNS:
        if col_name not in existing:
            conn.execute(f"ALTER TABLE recipes ADD COLUMN {col_name} {col_type}")
            added.append(col_name)

    if added:
        conn.commit()
        log.info(f"Đã thêm {len(added)} cột mới: {added}")
    else:
        log.info("Schema đã đầy đủ, không cần ALTER TABLE.")


# ──────────────────────────────────────────────
# 3. FETCH toàn bộ dish_ingredient × ingredients
# ──────────────────────────────────────────────
FETCH_SQL = """
SELECT
    di.recipe_id,
    di.quantity_g,
    di.is_main,
    i.thermogenic_score,
    i.hydration_score,
    i.warming_score,
    i.cooling_score,
    i.satiety_score,
    i.energy_density,
    i.sodium_density,
    i.glycemic_index,
    i.allergen_tags,
    i.is_animal_based,
    i.ingredient_type
FROM dish_ingredient di
JOIN ingredients_new i ON di.ingredient_id = i.id
WHERE di.quantity_g IS NOT NULL AND di.quantity_g > 0
"""

def fetch_all_dish_ingredients(conn: sqlite3.Connection) -> dict:
    """
    Trả về dict: { recipe_id: [ row_dict, ... ] }
    """
    cur = conn.execute(FETCH_SQL)
    cols = [d[0] for d in cur.description]

    data: dict = {}
    for row in cur.fetchall():
        r = dict(zip(cols, row))
        rid = r["recipe_id"]
        data.setdefault(rid, []).append(r)

    log.info(f"Loaded {sum(len(v) for v in data.values())} ingredient rows "
             f"cho {len(data)} recipes.")
    return data


# ──────────────────────────────────────────────
# 4. COMPUTE aggregated values cho 1 recipe
# ──────────────────────────────────────────────

# Danh sách ingredient_type coi là "vegetarian" (không ăn thịt nhưng ăn trứng/sữa)
# Điều chỉnh theo data thực tế của bạn
VEGETARIAN_ANIMAL_TYPES = {"egg", "dairy", "trứng", "sữa", "phô mai"}

def compute_recipe_agg(rows: list[dict]) -> dict:
    """
    rows: danh sách ingredient rows của 1 recipe (quantity_g > 0 đã lọc)
    """
    total_qty = sum(r["quantity_g"] for r in rows)

    if total_qty == 0:
        return {}

    def safe(val):
        """Trả về 0 nếu None."""
        return val if val is not None else 0.0

    # ── Weighted average helper ──
    def wavg(field: str) -> float | None:
        valid = [(r["quantity_g"], safe(r[field]))
                 for r in rows if r[field] is not None]
        if not valid:
            return None
        w_sum = sum(q for q, _ in valid)
        return round(sum(q * v for q, v in valid) / w_sum, 4) if w_sum else None

    # ── dish_energy_total: SUM(qty × energy_density / 100) ──
    energy_total = sum(
        r["quantity_g"] * safe(r["energy_density"]) / 100
        for r in rows if r["energy_density"] is not None
    )

    # ── dish_sodium_total: SUM(qty × sodium_density / 100) ──
    sodium_total = sum(
        r["quantity_g"] * safe(r["sodium_density"]) / 100
        for r in rows if r["sodium_density"] is not None
    )

    # ── dish_glycemic_load: SUM(GI × qty / 100) ──
    # NOTE: Thiếu carb_ratio vì schema không có → dùng xấp xỉ GI × qty/100
    # Kết quả mang tính tương đối, dùng để so sánh giữa các món
    glycemic_load = sum(
        safe(r["glycemic_index"]) * r["quantity_g"] / 100
        for r in rows if r["glycemic_index"] is not None
    )

    # ── allergen_summary: hợp nhất tất cả allergen_tags ──
    all_allergens: set[str] = set()
    for r in rows:
        tags = r.get("allergen_tags")
        if tags:
            try:
                parsed = json.loads(tags) if tags.startswith("[") else [
                    t.strip() for t in tags.split(",") if t.strip()
                ]
                all_allergens.update(parsed)
            except (json.JSONDecodeError, AttributeError):
                pass
    allergen_summary = json.dumps(sorted(all_allergens), ensure_ascii=False)

    # ── is_vegan: True nếu KHÔNG có ingredient nào is_animal_based = 1 ──
    is_vegan = int(all(safe(r["is_animal_based"]) == 0 for r in rows))

    # ── is_vegetarian: vegan + cho phép trứng/sữa ──
    def is_veg_ingredient(r) -> bool:
        if safe(r["is_animal_based"]) == 0:
            return True
        itype = (r.get("ingredient_type") or "").lower().strip()
        return itype in VEGETARIAN_ANIMAL_TYPES

    is_vegetarian = int(all(is_veg_ingredient(r) for r in rows))

    return {
        "dish_thermogenic_score": wavg("thermogenic_score"),
        "dish_hydration_score":   wavg("hydration_score"),
        "dish_warming_score":     wavg("warming_score"),
        "dish_cooling_score":     wavg("cooling_score"),
        "dish_satiety_score":     wavg("satiety_score"),
        "dish_energy_total":      round(energy_total, 2),
        "dish_sodium_total":      round(sodium_total, 2),
        "dish_glycemic_load":     round(glycemic_load, 4),
        "allergen_summary":       allergen_summary,
        "is_vegan":               is_vegan,
        "is_vegetarian":          is_vegetarian,
    }


# ──────────────────────────────────────────────
# 5. BATCH UPDATE vào recipes
# ──────────────────────────────────────────────
UPDATE_SQL = """
UPDATE recipes SET
    dish_thermogenic_score = :dish_thermogenic_score,
    dish_hydration_score   = :dish_hydration_score,
    dish_warming_score     = :dish_warming_score,
    dish_cooling_score     = :dish_cooling_score,
    dish_satiety_score     = :dish_satiety_score,
    dish_energy_total      = :dish_energy_total,
    dish_sodium_total      = :dish_sodium_total,
    dish_glycemic_load     = :dish_glycemic_load,
    allergen_summary       = :allergen_summary,
    is_vegan               = :is_vegan,
    is_vegetarian          = :is_vegetarian
WHERE id = :recipe_id
"""

def batch_update(conn: sqlite3.Connection, dish_data: dict, batch_size: int = 500):
    params_list = []
    skipped = 0

    for recipe_id, rows in dish_data.items():
        agg = compute_recipe_agg(rows)
        if not agg:
            skipped += 1
            continue
        agg["recipe_id"] = recipe_id
        params_list.append(agg)

    total = len(params_list)
    log.info(f"Chuẩn bị UPDATE {total} recipes (bỏ qua {skipped} do thiếu data).")

    for i in range(0, total, batch_size):
        batch = params_list[i : i + batch_size]
        conn.executemany(UPDATE_SQL, batch)
        conn.commit()
        log.info(f"  ✓ {min(i + batch_size, total)}/{total} recipes updated.")

    log.info("✅ Hoàn tất batch update.")


# ──────────────────────────────────────────────
# 6. VERIFICATION – in ra vài dòng mẫu
# ──────────────────────────────────────────────
VERIFY_SQL = """
SELECT id, title,
       dish_energy_total, dish_sodium_total,
       dish_warming_score, dish_cooling_score,
       is_vegan, allergen_summary
FROM recipes
WHERE dish_energy_total IS NOT NULL
LIMIT 5
"""

def verify(conn: sqlite3.Connection):
    cur = conn.execute(VERIFY_SQL)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print("\n── Sample kết quả ──")
    print(" | ".join(cols))
    print("-" * 100)
    for row in rows:
        print(" | ".join(str(v)[:20] if v is not None else "NULL" for v in row))


# ──────────────────────────────────────────────
# 7. MAIN  (Jupyter-friendly — không dùng argparse)
# ──────────────────────────────────────────────

# ✏️  CHỈ CẦN SỬA 3 DÒNG NÀY
DB_PATH    = "../model_store/cookpad.db"  # ← đường dẫn DB của bạn
DRY_RUN    = False                         # ← True: chỉ xem, KHÔNG ghi DB
BATCH_SIZE = 500

def main():
    db_path = Path(DB_PATH)
    if not db_path.exists():
        log.error(f"Không tìm thấy file DB: {db_path}")
        return

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    try:
        if not DRY_RUN:
            migrate_schema(conn)

        dish_data = fetch_all_dish_ingredients(conn)

        if DRY_RUN:
            for rid, rows in list(dish_data.items())[:3]:
                agg = compute_recipe_agg(rows)
                print(f"\nRecipe {rid}:")
                for k, v in agg.items():
                    print(f"  {k}: {v}")
        else:
            batch_update(conn, dish_data, batch_size=BATCH_SIZE)
            verify(conn)

    finally:
        conn.close()


main()